# N01. Do modelo por paciente à avaliação, com dados sintéticos

**Algoritmos exercitados:** P1 (modelo de estados por paciente), A1 (Kaplan-Meier), A2 (índice C).

**Ideia.** Sorteamos pacientes a partir de taxas *conhecidas* e verificamos se as ferramentas de avaliação recuperam o que sabemos ser verdade. É o mesmo princípio do estudo de simulação do Artigo 1 (C7), aplicado ao Artigo 2.

**Sem dados do registro.** Este notebook não lê nenhum arquivo do RHC/SP; pode ficar público no GitHub.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('../projetos/artigo2_caixacinza/codigo'))
os.environ.setdefault('NTHREADS', '1')
import numpy as np, torch, core
rng = np.random.default_rng(1)

## 1. Dois grupos de pacientes com taxas conhecidas

Grupo de baixo risco (EC1) e de alto risco (EC4). Todas as 16 taxas valem 2% ao mês, exceto o início do tratamento ($\gamma = 0{,}4$) e o óbito no estadiamento 4 ($\mu_4 = 0{,}10$). `simula_xi` dá a ocupação esperada dos estados mês a mês; `saidas_paciente` dá a sobrevida $S$ e o óbito no mês $O$.

In [2]:
H = 60
th = torch.full((2, core.K), 0.02); th[:, core.IX['gamma']] = 0.4; th[:, core.IX['mu_4']] = 0.10
ec = torch.tensor([1, 4]); Dc = torch.tensor([66, 66])
xi = core.simula_xi(th, ec, Dc, H); S, O, _, _ = core.saidas_paciente(th, xi)
S = S.numpy(); O = O.numpy()
print('S(60) verdadeira: EC1 = %.3f, EC4 = %.3f' % (S[0, 60], S[1, 60]))

S(60) verdadeira: EC1 = 0.288, EC4 = 0.034


## 2. Sorteio de pacientes

Para cada paciente, sorteamos o mês do óbito pela distribuição $O(t)$ do seu grupo; quem não morre até 60 meses é censurado em 60. Também censuramos 20% dos pacientes num mês uniforme, como perda de seguimento.

In [3]:
def sorteia(n, Sg, Og):
    p = np.r_[Og[1:H + 1], Sg[H]]                 # óbito em 1..H, ou vivo em H
    t = rng.choice(np.arange(1, H + 2), size=n, p=p / p.sum())
    d = (t <= H).astype(int); t = np.minimum(t, H)
    perde = rng.random(n) < .2; tc = rng.integers(1, H + 1, n)
    d = np.where(perde & (tc < t), 0, d); t = np.where(perde & (tc < t), tc, t)
    return t, d
n = 3000
t1, d1 = sorteia(n, S[0], O[0]); t4, d4 = sorteia(n, S[1], O[1])

## 3. A1: o Kaplan-Meier recupera a sobrevida verdadeira?

Esperamos diferença de poucos pontos (erro de amostragem com 3.000 pacientes).

In [4]:
km1 = core.kaplan_meier(t1, d1, H); km4 = core.kaplan_meier(t4, d4, H)
for nome, km, s in (('EC1', km1, S[0]), ('EC4', km4, S[1])):
    print(f'{nome}: KM S(60) = {km[60]:.3f}   verdadeira = {s[60]:.3f}   diferença = {100*(km[60]-s[60]):+.1f} pontos')

EC1: KM S(60) = 0.288   verdadeira = 0.288   diferença = +0.0 pontos
EC4: KM S(60) = 0.037   verdadeira = 0.034   diferença = +0.4 pontos


## 4. A2: o índice C separa os grupos?

Usamos como risco o estadiamento (1 ou 4). Com dois grupos, o índice C mede quanto o grupo de alto risco morre antes. Embaralhar o risco deve dar cerca de 0,5.

In [5]:
tau = np.r_[t1, t4]; dd = np.r_[d1, d4]; risco = np.r_[np.ones(n), 4 * np.ones(n)]
print('C com o estadiamento  = %.3f' % core.c_index(tau, dd, risco, max_n=4000))
print('C com risco embaralhado = %.3f' % core.c_index(tau, dd, rng.permutation(risco), max_n=4000))

C com o estadiamento  = 0.643
C com risco embaralhado = 0.508


## 5. Como usar este notebook como modelo

1. Um notebook por família de algoritmos, com dados sintéticos e verdade conhecida.
2. Cada seção diz o que se espera *antes* do resultado.
3. O que precisa valer sempre vira um teste automático em `testes/` (aqui: `test_A1`, `test_A2`, `test_P1`).
4. Notebooks são salvos **executados**, para que se leia o resultado no GitHub sem rodar.